# Image Filtering & Convolution Basics
## Computer Vision Practice - Week 2, Day 2 (Tuesday)

**Objective:** Master fundamental image filtering concepts and understand convolution from first principles.

**Topics:**
- Convolution operation fundamentals
- Kernel (filter) concept
- Common filter types (Blur, Sharpen, Gradient)
- Padding and stride concepts
- 2D convolution implementation from scratch
- OpenCV filtering functions

## 1. Libraries & Setup

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy import signal
import warnings
warnings.filterwarnings('ignore')

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ Libraries loaded!")

## 2. What is Convolution?

**Convolution** is the fundamental operation in image processing and deep learning.

**Simple Explanation:**
- Take a small **kernel** (filter) - a grid of numbers
- Slide it over the image
- At each position, multiply overlapping values and sum them
- This produces a new image with filtered results

**Why use convolution?**
- Blur an image (smooth out noise)
- Sharpen details
- Detect edges
- Extract features
- All the operations in deep learning!

**Visual analogy:** Imagine scanning a document with a scanner head - that's what convolution does!

## 3. Common Kernels (Filters)

**Box/Average Filter:**
- Average all neighbors
- Effect: Blurs the image
- Kernel: All values = 1/9
```
[1/9  1/9  1/9]
[1/9  1/9  1/9]
[1/9  1/9  1/9]
```

**Gaussian Blur:**
- Weighted average (center has more weight)
- Effect: Smooth blur, more natural
- Kernel: Bell curve pattern

**Sharpen Filter:**
- Enhances differences
- Effect: Makes edges stand out
- Kernel: Negative values around center

**Sobel Filter:**
- Detects edges
- Effect: Shows where intensity changes
- Kernel: Directional gradient

In [ ]:
# --- MORNING SESSION: Manual Convolution Example ---

# Create a simple test image
test_image = np.array([[1, 2, 3, 4, 5],
                       [6, 7, 8, 9, 10],
                       [11, 12, 13, 14, 15],
                       [16, 17, 18, 19, 20],
                       [21, 22, 23, 24, 25]], dtype=np.float32)

# Define a simple 3x3 kernel (box blur)
kernel = np.array([[1, 1, 1],
                   [1, 1, 1],
                   [1, 1, 1]], dtype=np.float32) / 9

print("Original Image:")
print(test_image.astype(int))
print(f"\nKernel (Box Blur):\n{kernel}")

# Apply convolution using scipy
convolved = signal.convolve2d(test_image, kernel, mode='same')

print(f"\nConvolved Image (with padding):")
print(convolved.astype(int))

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

axes[0].imshow(test_image, cmap='gray')
axes[0].set_title('Original Image', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(kernel, cmap='hot')
axes[1].set_title('Kernel (Filter)', fontweight='bold')
axes[1].axis('off')

axes[2].imshow(convolved, cmap='gray')
axes[2].set_title('Convolution Result (Blurred)', fontweight='bold')
axes[2].axis('off')

plt.suptitle('Manual Convolution Example', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- AFTERNOON SESSION: Real Image Filtering ---

# Create a sample image with noise to demonstrate filtering
sample_img = np.zeros((200, 300, 3), dtype=np.uint8)
cv2.rectangle(sample_img, (50, 50), (150, 150), (255, 0, 0), -1)
cv2.circle(sample_img, (200, 100), 40, (0, 255, 0), -1)
cv2.rectangle(sample_img, (200, 50), (280, 130), (0, 0, 255), -1)

# Add noise
noise = np.random.normal(0, 25, sample_img.shape).astype(np.uint8)
noisy_img = cv2.add(sample_img, noise)
noisy_gray = cv2.cvtColor(noisy_img, cv2.COLOR_BGR2GRAY)

# Define different kernels
box_kernel = np.ones((5, 5)) / 25
sharpen_kernel = np.array([[-1, -1, -1],
                           [-1, 9, -1],
                           [-1, -1, -1]])

# Apply different filters
blurred = cv2.blur(noisy_gray, (5, 5))
gaussian = cv2.GaussianBlur(noisy_gray, (5, 5), 1.0)
median = cv2.medianBlur(noisy_gray, 5)
sharpened = cv2.filter2D(noisy_gray, -1, sharpen_kernel)

# Visualize
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0, 0].imshow(noisy_gray, cmap='gray')
axes[0, 0].set_title('Original (with noise)', fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(blurred, cmap='gray')
axes[0, 1].set_title('Box Blur (5x5)', fontweight='bold')
axes[0, 1].axis('off')

axes[0, 2].imshow(gaussian, cmap='gray')
axes[0, 2].set_title('Gaussian Blur', fontweight='bold')
axes[0, 2].axis('off')

axes[1, 0].imshow(median, cmap='gray')
axes[1, 0].set_title('Median Blur', fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(sharpened, cmap='gray')
axes[1, 1].set_title('Sharpened', fontweight='bold')
axes[1, 1].axis('off')

# Bilateral filter (edge-preserving)
bilateral = cv2.bilateralFilter(noisy_gray, 9, 75, 75)
axes[1, 2].imshow(bilateral, cmap='gray')
axes[1, 2].set_title('Bilateral Filter', fontweight='bold')
axes[1, 2].axis('off')

plt.suptitle('Different Filtering Techniques', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✓ Different filters applied!")
print("- Box Blur: Simple average (fast but may blur edges)")
print("- Gaussian: Weighted average (smooth, natural)")
print("- Median: Middle value (great for salt-and-pepper noise)")
print("- Sharpen: Enhances edges")
print("- Bilateral: Blurs while preserving edges (best quality)")

## 4. Key Concepts Summary

**Padding:** Add extra border to image (zeros or reflection)
- Same: Output size = Input size (useful for preserving dimensions)
- Valid: Output size = (Input - Kernel) + 1 (loses border information)

**Stride:** How much to move kernel each step
- Stride=1: Move by 1 pixel (standard)
- Stride=2: Move by 2 pixels (reduces output size)

**Common OpenCV Functions:**
- `cv2.blur()` - Box filter
- `cv2.GaussianBlur()` - Gaussian filter
- `cv2.medianBlur()` - Median filter
- `cv2.bilateralFilter()` - Edge-preserving blur
- `cv2.filter2D()` - Apply custom kernel
- `cv2.morphologyEx()` - Morphological operations

**When to use which:**
- **Gaussian Blur**: General smoothing, reduce noise
- **Median Blur**: Salt-and-pepper noise
- **Bilateral**: When you need to keep edges sharp
- **Sharpen**: Enhance details